## Análisis de datos II - Clase 3
---

### Análisis de calidad de datos
---

In [101]:
import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import missingno as msno

1. Dataset original

In [102]:
penguins = sns.load_dataset("penguins")
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [103]:
penguins.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


In [104]:
f_numericas = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
f_categoricas = ['species', 'island', 'sex']

2. Simulamos un dataset con problemas de calidad ("sucio")

In [105]:
penguins_sucio = penguins.copy()

# Completitud: aumento los nulos en bill_length_mm a 15% de la muestra
idx_nulos = penguins_sucio.sample(frac=0.15, random_state=1).index
penguins_sucio.loc[idx_nulos, "bill_length_mm"] = np.nan

# Validez: agrego un valor imposible
penguins_sucio.loc[penguins_sucio.index[0], "body_mass_g"] = -500

# Validez: agrego una categoría no reconocida (pingüino emperador)
penguins_sucio.loc[penguins_sucio.index[10], "species"] = "Emperor"

# Unicidad: agrego filas duplicadas
penguins_sucio = pd.concat([penguins_sucio, penguins_sucio.sample(5, random_state=2)], ignore_index=True)

# Consistencia: agrego un valor inconsistente
penguins_sucio.loc[penguins_sucio.index[20], "bill_length_mm"] = 5.0 

penguins_sucio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 349 entries, 0 to 348
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            349 non-null    object 
 1   island             349 non-null    object 
 2   bill_length_mm     294 non-null    float64
 3   bill_depth_mm      347 non-null    float64
 4   flipper_length_mm  347 non-null    float64
 5   body_mass_g        347 non-null    float64
 6   sex                338 non-null    object 
dtypes: float64(4), object(3)
memory usage: 19.2+ KB


In [106]:
for _, cat in enumerate(f_categoricas):
    penguins[cat] = penguins[cat].astype('category')

for _, cat in enumerate(f_categoricas):
    penguins_sucio[cat] = penguins_sucio[cat].astype('category')

In [107]:
print("ESTADÍSTICAS DESCRIPTIVAS DEL DATASET ORIGINAL:")
penguins.describe(include='all')

ESTADÍSTICAS DESCRIPTIVAS DEL DATASET ORIGINAL:


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
count,344,344,342.000000,342.000000,342.000000,342.000000,333
unique,3,3,NaN,NaN,NaN,NaN,2
top,Adelie,Biscoe,NaN,NaN,NaN,NaN,Male
freq,152,168,NaN,NaN,NaN,NaN,168
mean,NaN,NaN,43.921930,17.151170,200.915205,4201.754386,NaN
std,NaN,NaN,5.459584,1.974793,14.061714,801.954536,NaN
min,NaN,NaN,32.100000,13.100000,172.000000,2700.000000,NaN
25%,NaN,NaN,39.225000,15.600000,190.000000,3550.000000,NaN
50%,NaN,NaN,44.450000,17.300000,197.000000,4050.000000,NaN
75%,NaN,NaN,48.500000,18.700000,213.000000,4750.000000,NaN


In [108]:
print("INFO DEL DATASET SUCIO:\n")
penguins_sucio.info()

INFO DEL DATASET SUCIO:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 349 entries, 0 to 348
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   species            349 non-null    category
 1   island             349 non-null    category
 2   bill_length_mm     294 non-null    float64 
 3   bill_depth_mm      347 non-null    float64 
 4   flipper_length_mm  347 non-null    float64 
 5   body_mass_g        347 non-null    float64 
 6   sex                338 non-null    category
dtypes: category(3), float64(4)
memory usage: 12.5 KB


In [109]:
print("ESTADÍSTICAS DESCRIPTIVAS DEL DATASET SUCIO:")
penguins_sucio.describe(include='all')

ESTADÍSTICAS DESCRIPTIVAS DEL DATASET SUCIO:


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
count,349,349,294.000000,347.000000,347.000000,347.000000,338
unique,4,3,NaN,NaN,NaN,NaN,2
top,Adelie,Biscoe,NaN,NaN,NaN,NaN,Male
freq,154,171,NaN,NaN,NaN,NaN,172
mean,NaN,NaN,43.823469,17.167147,200.858790,4190.634006,NaN
std,NaN,NaN,5.870371,1.971257,14.105925,838.946853,NaN
min,NaN,NaN,5.000000,13.100000,172.000000,-500.000000,NaN
25%,NaN,NaN,39.350000,15.600000,190.000000,3550.000000,NaN
50%,NaN,NaN,44.700000,17.300000,197.000000,4050.000000,NaN
75%,NaN,NaN,48.400000,18.700000,213.000000,4750.000000,NaN


---
### Chequeos de calidad
---
#### Implementación de chequeos de calidad sobre el dataset modificado de los Pingüinos (vamos a verificar todas las dimensiones excepto Timeliness y Accuracy que se mostrarán con otros datasets)

In [110]:
import great_expectations as gx

In [111]:
# Definimos el contexto de Great Expectations

context = gx.get_context()
data_source = context.data_sources.add_pandas("pandas_source") # Le aviso que voy a trabajar con formato de pandas
data_asset = data_source.add_dataframe_asset(name="penguins_asset") # Defino un asset - un dataset
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch_def") # Cada vez que se pida un batch de este dataset, que se tome todo el dataset
batch = batch_definition.get_batch(batch_parameters={"dataframe": penguins_sucio}) # Crear el batch

In [ ]:
#1. Completitud

print("Nulos:")
tasa_nulos_original= penguins["bill_length_mm"].isna().sum() / len(penguins) * 100
tasa_nulos_actual = penguins_sucio["bill_length_mm"].isna().sum() / len(penguins) * 100
print(f"Tasa de nulos original: {tasa_nulos_original:.1f}%")
print(f"Tasa de nulos actual: {tasa_nulos_actual:.1f}%")

#Ejecuto el check
resultado_completitud = batch.validate(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="bill_length_mm", mostly=0.90)
)
print(f"Completitud (bill_length_mm): {'OK' if resultado_completitud.success else 'FAIL'}")


Nulos
Tasa de nulos original: 0.6%
Tasa de nulos actual: 16.0%


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Completitud (bill_length_mm): FAIL


In [113]:
#2. Validez — estructura/esquema, rango de valores y categorías

resultado_columnas = batch.validate(
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=[
            "species", "island", "bill_length_mm", "bill_depth_mm",
            "flipper_length_mm", "body_mass_g", "sex"
        ]
    )
)
print(f"Esquema (columnas esperadas): {'OK' if resultado_columnas.success else 'FAIL'}")

resultado_rango = batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(column="body_mass_g", min_value=2000, max_value=7000)
)
print(f"Validez (body_mass_g en rango): {'OK' if resultado_rango.success else 'FAIL'} "
      f"({resultado_rango.result['unexpected_count']} valores fuera de rango)")

resultado_categorias = batch.validate(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="species", value_set=["Adelie", "Chinstrap", "Gentoo"]
    )
)
print(f"Validez (categorías de species): {'OK' if resultado_categorias.success else 'FAIL'} "
      f"({resultado_categorias.result['unexpected_count']} categorías no reconocidas)")


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Esquema (columnas esperadas): OK


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Validez (body_mass_g en rango): FAIL (1 valores fuera de rango)


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Validez (categorías de species): FAIL (1 categorías no reconocidas)


In [114]:
# 3. Unicidad

resultado_unicidad = batch.validate(
    gx.expectations.ExpectCompoundColumnsToBeUnique(
        column_list=["species", "island", "bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
    )
)
print(f"Unicidad: {'OK' if resultado_unicidad.success else 'FAIL'} "
      f"({resultado_unicidad.result['unexpected_count']} duplicados)")


Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

Unicidad: FAIL (12 duplicados)


In [115]:
#4. Consistencia 

penguins_sucio["diferencia_bill"] = penguins_sucio["bill_length_mm"] - penguins_sucio["bill_depth_mm"]

data_asset_2 = data_source.add_dataframe_asset(name="penguins_consistencia")
batch_def_2 = data_asset_2.add_batch_definition_whole_dataframe("batch_def_2")
batch_2 = batch_def_2.get_batch(batch_parameters={"dataframe": penguins_sucio})

resultado_consistencia = batch_2.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(column="diferencia_bill", min_value=0, mostly=0.99)
)
print(f"Consistencia (bill_length > bill_depth): {'OK' if resultado_consistencia.success else 'FAIL'}")

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Consistencia (bill_length > bill_depth): OK


#### Timeliness - ejemplo con serie temporal

Para este ejemplo usamos el dataset [sensor data de Kaggle](https://www.kaggle.com/datasets/marcpaulo/sensor-data?resource=download) 

In [116]:
sensores = pd.read_csv("../datasets/sensor_data.csv")
sensores["time"] = pd.to_datetime(sensores["time"], format="%H:%M")
sensores.head()

,time,SensorA,SensorB,SensorC
0,1900-01-01 00:00:00,1.416250,4.219930,3.139646
1,1900-01-01 00:01:00,3.534439,NaN,3.064088
2,1900-01-01 00:02:00,5.659733,NaN,2.925565
3,1900-01-01 00:03:00,5.640167,3.073650,5.381210
4,1900-01-01 00:04:00,4.454474,1.554044,1.055965


In [117]:
sensores.shape

(1440, 4)

In [118]:

# Simulamos un problema real: se cortó la transmisión del sensor por un rato
sensores_con_gap = sensores.drop(sensores.index[300:320]).reset_index(drop=True) # Saco 20 minutos seguidos de lecturas
sensores_con_gap["time_numeric"] = sensores_con_gap["time"].astype("int64") # Lo convierto a int64 para poder usarlo con GX (no opera bien con datetime)


In [119]:
# Contexto de GX

context = gx.get_context()
data_source = context.data_sources.add_pandas("pandas_source_timeliness")
data_asset = data_source.add_dataframe_asset(name="sensores_asset_timeliness")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch_def_timeliness")
batch = batch_definition.get_batch(batch_parameters={"dataframe": sensores_con_gap})


In [120]:

# 1. Chequeo de orden cronológico

resultado_orden = batch.validate(
    gx.expectations.ExpectColumnValuesToBeIncreasing(column="time_numeric")
)
print(f"Orden cronológico: {'OK' if resultado_orden.success else 'FAIL'}")


# 2. Completitud dentro de una ventana temporal (es combinación de Completitud + Timeliness)
# Ejemplo: para predecir necesito las últimas 10 lecturas.

ahora_simulado = sensores_con_gap["time"].max()
ventana_minutos = 10
inicio_ventana = ahora_simulado - pd.Timedelta(minutes=ventana_minutos)

ventana_reciente = sensores_con_gap[
    (sensores_con_gap["time"] > inicio_ventana) & (sensores_con_gap["time"] <= ahora_simulado)
]

muestras_esperadas = len(pd.date_range(start=inicio_ventana, end=ahora_simulado, freq="1min"))
print(f"Ventana analizada: {inicio_ventana.time()} - {ahora_simulado.time()}")
print(f"Muestras esperadas en la ventana: {muestras_esperadas}")


data_asset_ventana = data_source.add_dataframe_asset(name="ventana_reciente_asset")
batch_def_ventana = data_asset_ventana.add_batch_definition_whole_dataframe("batch_def_ventana")
batch_ventana = batch_def_ventana.get_batch(batch_parameters={"dataframe": ventana_reciente})

resultado_ventana = batch_ventana.validate(
    gx.expectations.ExpectTableRowCountToEqual(value=muestras_esperadas)
)
print(f"Completitud de la ventana: {'OK' if resultado_ventana.success else 'FAIL: faltan muestras para predecir.'} "
      f"Se encontraron {len(ventana_reciente)} de {muestras_esperadas} esperadas")


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Orden cronológico: OK
Ventana analizada: 23:49:00 - 23:59:00
Muestras esperadas en la ventana: 11


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Completitud de la ventana: FAIL: faltan muestras para predecir. Se encontraron 10 de 11 esperadas


In [121]:
resumen_timeliness = pd.DataFrame([
    {"chequeo": "Orden cronológico", "pasa_el_test": resultado_orden.success},
    {"chequeo": "Completitud de ventana de datos", "pasa_el_test": resultado_ventana.success},
])
resumen_timeliness["estado"] = np.where(resumen_timeliness["pasa_el_test"], "OK", "ALERTA")
resumen_timeliness

,chequeo,pasa_el_test,estado
0,Orden cronológico,True,OK
1,Completitud de ventana de datos,False,ALERTA


#### Chequeo de accuracy (exactitud) - ejemplo
#### Vamos a usar datos ficticios de nacimientos por comuna en CABA, y vamos a validar el dato de las comunas contra una [lista externa](https://data.buenosaires.gob.ar/dataset/comunas/resource/Juqdkmgo-612221-resource-xlsx/download) tomada de la página del gobierno de la Ciudad

In [122]:
comunas_oficial = pd.read_csv(
    "../datasets/comunas.csv",
    sep=";",
    encoding="utf-8-sig"
)
comunas_oficial[["id", "comuna", "barrios"]]

,id,comuna,barrios
0,1,1,"Constitucion, San Telmo, Monserrat, Retiro, Pu..."
1,2,2,Recoleta
2,3,3,"San Cristobal, Balvanera"
3,4,4,"Barracas, La Boca, Parque Patricios, Nueva Pom..."
4,5,5,"Almagro, Boedo"
5,6,6,Caballito
6,7,7,"Flores, Parque Chacabuco"
7,8,8,"Villa Lugano, Villa Riachuelo, Villa Soldati"
8,9,9,"Parque Avellaneda, Mataderos, Liniers"
9,10,10,"Floresta, Monte Castro, Velez Sarsfield, Versa..."


In [123]:
# Armo el diccionario de referencia: barrio y su comuna
barrio_y_comuna = {}
for _, row in comunas_oficial.iterrows():
    for barrio in row["barrios"].split(","):
        barrio_y_comuna[barrio.strip()] = row["comuna"]

barrio_y_comuna

{'Constitucion': 1,
 'San Telmo': 1,
 'Monserrat': 1,
 'Retiro': 1,
 'Puerto Madero': 1,
 'San Nicolas': 1,
 'Recoleta': 2,
 'San Cristobal': 3,
 'Balvanera': 3,
 'Barracas': 4,
 'La Boca': 4,
 'Parque Patricios': 4,
 'Nueva Pompeya': 4,
 'Almagro': 5,
 'Boedo': 5,
 'Caballito': 6,
 'Flores': 7,
 'Parque Chacabuco': 7,
 'Villa Lugano': 8,
 'Villa Riachuelo': 8,
 'Villa Soldati': 8,
 'Parque Avellaneda': 9,
 'Mataderos': 9,
 'Liniers': 9,
 'Floresta': 10,
 'Monte Castro': 10,
 'Velez Sarsfield': 10,
 'Versalles': 10,
 'Villa Luro': 10,
 'Villa Real': 10,
 'Villa Del Parque': 11,
 'Villa Devoto': 11,
 'Villa Gral. Mitre': 11,
 'Villa Santa Rita': 11,
 'Villa Urquiza': 12,
 'Villa Pueyrredon': 12,
 'Saavedra': 12,
 'Coghlan': 12,
 'Belgrano': 13,
 'Nuñez': 13,
 'Colegiales': 13,
 'Palermo': 14,
 'Villa Ortuzar': 15,
 'Agronomia': 15,
 'Paternal': 15,
 'Parque Chas': 15,
 'Chacarita': 15,
 'Villa Crespo': 15}

In [124]:
# Simulo el dataset de nacimientos. Tomo los barrios del diccionario de referencia y genero 200 registros aleatorios.

np.random.seed(42)

barrios_disponibles = list(barrio_y_comuna.keys())
n = 200

nacimientos = pd.DataFrame({
    "id_nacimiento": range(1, n + 1),
    "barrio": np.random.choice(barrios_disponibles, size=n),
})

nacimientos["comuna_declarada"] = nacimientos["barrio"].map(barrio_y_comuna)

## Simulo algunos errores de carga (barrio y comuna no coinciden)
idx_error = nacimientos.sample(frac=0.08, random_state=1).index
nacimientos.loc[idx_error, "comuna_declarada"] = np.random.randint(1, 16, size=len(idx_error))

nacimientos.head(10)

,id_nacimiento,barrio,comuna_declarada
0,1,Belgrano,13
1,2,Villa Luro,10
2,3,Boedo,5
3,4,Villa Ortuzar,15
4,5,San Cristobal,12
5,6,Villa Soldati,8
6,7,Belgrano,13
7,8,Villa Lugano,8
8,9,Mataderos,9
9,10,La Boca,4


In [125]:

# Agrego la columna de referencia para poder comparar con la declarada (lo requiere GX)
nacimientos["comuna_correcta"] = nacimientos["barrio"].map(barrio_y_comuna) 

# Contexto de GX
context = gx.get_context()
data_source = context.data_sources.add_pandas("pandas_source_accuracy")
data_asset = data_source.add_dataframe_asset(name="nacimientos_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch_def_accuracy")
batch = batch_definition.get_batch(batch_parameters={"dataframe": nacimientos})

In [126]:
resultado_accuracy = batch.validate(
    gx.expectations.ExpectColumnPairValuesToBeEqual(
        column_A="comuna_declarada",
        column_B="comuna_correcta"
    )
)
print(f"Accuracy (comuna declarada vs. comuna real según barrio): "
      f"{'OK' if resultado_accuracy.success else 'FAIL'} "
      f"— {resultado_accuracy.result['unexpected_count']} registros inexactos "
      f"({resultado_accuracy.result['unexpected_percent']:.1f}%)")

Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

Accuracy (comuna declarada vs. comuna real según barrio): FAIL — 16 registros inexactos (8.0%)


In [127]:
inexactos = nacimientos[nacimientos["comuna_declarada"] != nacimientos["comuna_correcta"]]
inexactos[["id_nacimiento", "barrio", "comuna_declarada", "comuna_correcta"]]

,id_nacimiento,barrio,comuna_declarada,comuna_correcta
4,5,San Cristobal,12,3
11,12,Liniers,1,9
18,19,Liniers,3,9
29,30,Velez Sarsfield,14,10
34,35,Chacarita,1,15
40,41,Balvanera,4,3
58,59,Flores,9,7
89,90,Paternal,5,15
95,96,Floresta,4,10
102,103,Liniers,14,9


In [128]:
resumen_accuracy = pd.DataFrame([
    {"chequeo": "Accuracy (comuna declarada vs. comuna real)", "pasa_el_test": resultado_accuracy.success},
])
resumen_accuracy["estado"] = np.where(resumen_accuracy["pasa_el_test"], "OK", "ALERTA")
resumen_accuracy

,chequeo,pasa_el_test,estado
0,Accuracy (comuna declarada vs. comuna real),False,ALERTA
